In [3]:
from google.colab import drive
from matplotlib import pyplot as plt
from tqdm import tqdm
import numpy as np
import pandas as pd

drive.mount('/content/drive')
df = pd.read_csv("/content/drive/My Drive/Model Training/model_training_data.csv")

df

Mounted at /content/drive


,month,-0_days_game_scheduled,-1_days_game_scheduled,-2_days_game_scheduled,-3_days_game_scheduled,-4_days_game_scheduled,-5_days_game_scheduled,-6_days_game_scheduled,-7_days_game_scheduled,-8_days_game_scheduled,...,-5_days_arena_long,-6_days_arena_long,-7_days_arena_long,-8_days_arena_long,-9_days_arena_long,-10_days_arena_long,-11_days_arena_long,-12_days_arena_long,-13_days_arena_long,injured?
0,Oct,1,0,0,0,0,0,0,0,0,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,False
1,Oct,0,1,0,0,0,0,0,0,0,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,False
2,Oct,1,0,1,0,0,0,0,0,0,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,False
3,Oct,1,1,0,1,0,0,0,0,0,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,False
4,Oct,0,1,1,0,1,0,0,0,0,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
491527,Apr,0,0,0,0,0,1,0,1,0,...,122.3875,0.0000,105.0075,105.0075,0.0000,0.0000,0.0000,105.0075,0.0000,False
491528,Apr,0,0,0,0,0,0,1,0,1,...,0.0000,122.3875,0.0000,105.0075,105.0075,0.0000,0.0000,0.0000,105.0075,False
491529,Apr,0,0,0,0,0,0,0,1,0,...,105.0075,0.0000,122.3875,0.0000,105.0075,105.0075,0.0000,0.0000,0.0000,False
491530,Apr,0,0,0,0,0,0,0,0,1,...,0.0000,105.0075,0.0000,122.3875,0.0000,105.0075,105.0075,0.0000,0.0000,False


In [4]:
lat_long_cols = list(df.columns)[13:41]
lat_cols = lat_long_cols[:14]

delete_flag = []
recent_four_lats = []
for idx in tqdm(df.index):
  lat_values = df.loc[idx, lat_cols]

  recent_vals = []
  for val in lat_values:
    if val != 0.0:
      if len(recent_vals) < 4:
        recent_vals.append(val)

  if len(recent_vals) == 4:
    delete_flag.append(0)
    recent_four_lats.append(recent_vals)
  else:
    delete_flag.append(1)


delete_col = pd.DataFrame({'delete_flag':delete_flag})
delete_col.index = df.index
df = pd.concat([df, delete_col], axis=1)
df = df[df['delete_flag'] == 0]
recent_lats = pd.DataFrame(recent_four_lats)
recent_lats.columns = ['1st_recent_lat', '2nd_recent_lat', '3rd_recent_lat', '4th_recent_lat']
recent_lats.index = df.index
df = pd.concat([df,recent_lats], axis=1)


arena = pd.read_csv('/content/drive/My Drive/Extracted Data Organized/arena_dim.csv')
gfact = pd.read_csv("/content/drive/My Drive/Extracted Data Organized/game_fact.csv")


#script converting cartesian coordinates of each arena from dtm to decimal
import re

def dms2dd(degrees, minutes, seconds, direction):
    dd = float(degrees) + float(minutes)/60 + float(seconds)/(60*60);
    if direction == 'E' or direction == 'S':
        dd *= -1
    return dd;

def dtm_to_dec(dms):
    parts = re.split('[^\d\w]+', dms)
    coor = dms2dd(parts[0], parts[1], parts[2], parts[3])

    return (coor)

for col in ['arena_lat', 'arena_long']:
  arena[col] = arena[col].apply(dtm_to_dec)


# region coding

arena_info = arena[['arena_name', 'arena_lat']]
arena_names = list(arena['arena_name'])
arena_region = [
    'NW', 'S', 'MW', 'SW', 'E', 'S', 'S', 'MW', 'MW', 'SW', 'S', 'NW', 'RM',
    'RM', 'E', 'S', 'MW', 'S', 'S', 'NW', 'NW', 'MW', 'S', 'E', 'E', 'MW',
    'SW', 'MW', 'E', 'MW', 'S', 'S', 'S']

arena_dict = {k:v for k,v in zip(arena_names, arena_region)}
arena_info['region'] = arena_info['arena_name'].apply(lambda x:arena_dict[x])


arena_info['arena_lat'] = round(arena_info['arena_lat'], 3)
for col in ['1st_recent_lat', '2nd_recent_lat', '3rd_recent_lat', '4th_recent_lat']:
  df[col] = round(df[col], 3)

region_code_dict = arena_info[['arena_lat', 'region']].set_index('arena_lat').to_dict()['region']
for col in ['1st_recent_lat', '2nd_recent_lat', '3rd_recent_lat', '4th_recent_lat']:
  df[col] = df[col].apply(lambda x:region_code_dict[x])
# cols to drop
lat_cols = list(df.columns)[13:27]
long_cols = list(df.columns)[27:41]
cols_to_delete = lat_cols + long_cols + ['delete_flag']

df = df.drop(cols_to_delete, axis=1)
df = df.rename(columns={
    '1st_recent_lat':'1st_recent_region',
    '2nd_recent_lat':'2nd_recent_region',
    '3rd_recent_lat':'3rd_recent_region',
    '4th_recent_lat':'4th_recent_region'})
df

<>:46: SyntaxWarning: invalid escape sequence '\d'
<>:46: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipython-input-578451224.py:46: SyntaxWarning: invalid escape sequence '\d'
  parts = re.split('[^\d\w]+', dms)
100%|██████████| 491532/491532 [03:51<00:00, 2121.88it/s]
/tmp/ipython-input-578451224.py:33: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  gfact = pd.read_csv("/content/drive/My Drive/Extracted Data Organized/game_fact.csv")
/tmp/ipython-input-578451224.py:65: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  arena_info['region'] = arena_info['arena_name'].apply(lambda x:arena_dict[x])
/tmp/ipython-input-578451224.py:68: SettingWithCopyWarning: 
A value is trying to be set on a c

,month,-0_days_game_scheduled,-1_days_game_scheduled,-2_days_game_scheduled,-3_days_game_scheduled,-4_days_game_scheduled,-5_days_game_scheduled,-6_days_game_scheduled,-7_days_game_scheduled,-8_days_game_scheduled,sum_field_goals_attempted,sum_minutes,sum_injury_history,injured?,1st_recent_region,2nd_recent_region,3rd_recent_region,4th_recent_region
5,Oct,1,0,1,1,0,1,0,0,0,11.0,37.0,106,False,SW,RM,S,NW
6,Oct,0,1,0,1,1,0,1,0,0,13.0,44.0,105,False,SW,RM,S,NW
7,Oct,0,0,1,0,1,1,0,1,0,13.0,44.0,104,False,SW,RM,S,NW
8,Oct,1,0,0,1,0,1,1,0,1,13.0,44.0,103,False,NW,SW,RM,S
9,Oct,0,1,0,0,1,0,1,1,0,16.0,51.0,102,False,NW,SW,RM,S
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
491527,Apr,0,0,0,0,0,1,0,1,0,19.0,101.0,76,False,NW,RM,NW,RM
491528,Apr,0,0,0,0,0,0,1,0,1,19.0,101.0,75,False,NW,RM,NW,RM
491529,Apr,0,0,0,0,0,0,0,1,0,19.0,101.0,74,False,RM,NW,RM,NW
491530,Apr,0,0,0,0,0,0,0,0,1,19.0,101.0,73,False,RM,NW,RM,NW


In [5]:
df.to_csv('model_training_data_le.csv', index=False)